In [ ]:
!pip install pypdf langchain_text_splitters

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
import numpy as np
import plotly.graph_objects as go
import umap
import json
import re
from pypdf import PdfReader
from google import genai
from sentence_transformers import CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
# Load environment variables
load_dotenv()


# STEP 1: CONFIGURATION

In [ ]:

# File to process
PDF_FILE = "/content/rag2020.pdf"  # ← CHANGE THIS TO YOUR PDF
from google.colab import userdata
# API Keys
GEMINI_API_KEY = userdata.get('Gemini_key') # or GEMINI_LLM_KEY

# Settings
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
TOP_K_RETRIEVE = 10
TOP_K_RERANK = 5
DISTANCE_THRESHOLD = 1.5

# Models
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
RERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
GEMINI_MODEL = "gemini-2.5-flash"

client = genai.Client(api_key=GEMINI_API_KEY)


print(f" Document: {PDF_FILE}")
print(f" Embedding Model: {EMBEDDING_MODEL}")
print(f" Reranker Model: {RERANKER_MODEL}")
print(f" LLM Model: {GEMINI_MODEL}")
print(f" Chunk Size: {CHUNK_SIZE} chars, Overlap: {CHUNK_OVERLAP}")


 Document: /content/rag2020.pdf
 Embedding Model: all-MiniLM-L6-v2
 Reranker Model: cross-encoder/ms-marco-MiniLM-L-6-v2
 LLM Model: gemini-2.5-flash
 Chunk Size: 1000 chars, Overlap: 200


# STEP 2: DOCUMENT LOADING (PyPDF)


In [ ]:

# Check if file exists
if not Path(PDF_FILE).exists():
    print(f"❌ File not found: {PDF_FILE}")
    print("Please update the PDF_FILE variable at the top of the script")
    exit()

# Load PDF
print(f"📖 Reading PDF: {PDF_FILE}")
pdf_reader = PdfReader(PDF_FILE)
total_pages = len(pdf_reader.pages)

# Extract text from all pages
full_text = ""
for page_num, page in enumerate(pdf_reader.pages, 1):
    page_text = page.extract_text()
    if page_text:
        full_text += page_text + "\n"
    print(f"  ✅ Processed page {page_num}/{total_pages}")

print(f"\n✅ Loaded {len(full_text):,} characters from {total_pages} pages")

📖 Reading PDF: /content/rag2020.pdf
  ✅ Processed page 1/19
  ✅ Processed page 2/19
  ✅ Processed page 3/19
  ✅ Processed page 4/19
  ✅ Processed page 5/19
  ✅ Processed page 6/19
  ✅ Processed page 7/19
  ✅ Processed page 8/19
  ✅ Processed page 9/19
  ✅ Processed page 10/19
  ✅ Processed page 11/19
  ✅ Processed page 12/19
  ✅ Processed page 13/19
  ✅ Processed page 14/19
  ✅ Processed page 15/19
  ✅ Processed page 16/19
  ✅ Processed page 17/19
  ✅ Processed page 18/19
  ✅ Processed page 19/19

✅ Loaded 69,075 characters from 19 pages


# STEP 3: CHUNKING (LangChain Text Splitter)


In [ ]:

# Create text splitter
print(f"Creating chunks (size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP})")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ".", "!", "?", " ", ""],
    length_function=len
)

# Split text into chunks
chunks = text_splitter.split_text(full_text)

print(f"Created {len(chunks)} chunks")
print(f"\n Sample chunks:")
for i, chunk in enumerate(chunks[:3], 1):
    preview = chunk[:100] + "..." if len(chunk) > 100 else chunk
    print(f"  Chunk {i}: {preview}")

Creating chunks (size=1000, overlap=200)
Created 88 chunks

 Sample chunks:
  Chunk 1: Retrieval-Augmented Generation for
Knowledge-Intensive NLP Tasks
Patrick Lewis†‡, Ethan Perez⋆,
Alek...
  Chunk 2: decisions and updating their world knowledge remain open research problems. Pre-
trained models with...
  Chunk 3: per token. We ﬁne-tune and evaluate our models on a wide range of knowledge-
intensive NLP tasks and...


# STEP 4: EMBEDDING (SentenceTransformer)

In [ ]:

# Load embedding model
print(f"Loading SentenceTransformer: {EMBEDDING_MODEL}")
embedding_model = SentenceTransformer(EMBEDDING_MODEL)

# Generate embeddings for all chunks
print(f"Generating embeddings for {len(chunks)} chunks...")
chunk_embeddings = embedding_model.encode(
    chunks,
    convert_to_numpy=True,
    show_progress_bar=True
)

print(f"Generated embeddings with shape: {chunk_embeddings.shape}")
print(f"Embedding dimension: {chunk_embeddings.shape[1]}")

Loading SentenceTransformer: all-MiniLM-L6-v2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generating embeddings for 88 chunks...


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Generated embeddings with shape: (88, 384)
Embedding dimension: 384


# STEP 5: QUERY ENHANCEMENT (Google Gemini)


In [ ]:

# Get query from user
query = input("\n❓ Enter your question: ").strip()

if not query:
    query = "What is this document about?"
    print(f"Using default query: {query}")

print(f"\n🔍 Original Query: {query}")

# Query Enhancement using Google Gemini
print("\n🤖 Enhancing query with Gemini...")

# Configure Gemini client
gemini_client = genai.Client(api_key=GEMINI_API_KEY)

# Query enhancement prompt
enhance_prompt = f"""You are a search-query expert for a RAG retrieval system working on technical and academic documents.

Given the user question below, output a single JSON object with these keys:

"sub_queries"  — array of exactly 3 short, self-contained questions that together cover every important aspect of the original question (e.g. definition, mechanism, evidence, comparison, limitations). Each must be independently searchable.

"hyde"         — a 2-3 sentence hypothetical passage written as if it appeared in a relevant technical or academic document that directly answers the question. Use precise domain vocabulary. This passage will be embedded and compared against document chunks, so it must sound like document text, not a question.

"step_back"    — ONE broader question that captures the underlying principle or concept behind the original question.

Rules:
- Output ONLY valid JSON. No markdown fences, no preamble, no extra keys.
- "sub_queries" → array of strings
- "hyde"        → single string
- "step_back"   → single string

Example:
{{
    "sub_queries": ["What is X?", "How does X improve Y?", "What are X's limitations?"],
    "hyde": "X is a technique that achieves Y through Z. Empirical results show...",
    "step_back": "What are the general principles of X-based approaches?"
}}

User question: {query}"""

# Generate enhanced queries
try:
    response = gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=enhance_prompt
    )

    raw_response = response.text.strip()

    # Clean response
    raw_response = re.sub(r"^```[a-zA-Z]*\s*|\s*```$", "", raw_response, flags=re.MULTILINE).strip()

    # Extract JSON
    match = re.search(r"\{.*\}", raw_response, re.DOTALL)
    if match:
        raw_response = match.group(0)

    # Remove trailing commas
    raw_response = re.sub(r",\s*}", "}", raw_response)
    raw_response = re.sub(r",\s*]", "]", raw_response)

    # Parse JSON
    parsed = json.loads(raw_response)

    # Build enhanced queries list
    enhanced_queries = [query]  # Original query first

    # Add sub-queries
    for sub_q in parsed.get("sub_queries", []):
        sub_q = str(sub_q).strip()
        if sub_q and sub_q not in enhanced_queries:
            enhanced_queries.append(sub_q)

    # Add HyDE
    hyde = parsed.get("hyde", "").strip()
    if hyde and hyde not in enhanced_queries:
        enhanced_queries.append(hyde)

    # Add step-back
    step_back = parsed.get("step_back", "").strip()
    if step_back and step_back not in enhanced_queries:
        enhanced_queries.append(step_back)

    print(f"✅ Generated {len(enhanced_queries)} enhanced queries:")
    labels = ["Original", "Sub-Query 1", "Sub-Query 2", "Sub-Query 3", "HyDE", "Step-Back"]
    for i, eq in enumerate(enhanced_queries):
        label = labels[i] if i < len(labels) else f"Query {i+1}"
        preview = eq[:80] + "..." if len(eq) > 80 else eq
        print(f"  [{label}] {preview}")

except Exception as e:
    print(f"⚠️ Query enhancement failed: {e}")
    print("Continuing with original query only...")
    enhanced_queries = [query]


❓ Enter your question: what is rag

🔍 Original Query: what is rag

🤖 Enhancing query with Gemini...
✅ Generated 6 enhanced queries:
  [Original] what is rag
  [Sub-Query 1] What is Retrieval-Augmented Generation (RAG)?
  [Sub-Query 2] How does RAG integrate retrieval with large language models?
  [Sub-Query 3] What are the primary benefits of using RAG in AI systems?
  [HyDE] Retrieval-Augmented Generation (RAG) is an architectural pattern designed to enh...
  [Step-Back] What are the architectural approaches to augment generative AI models with exter...


# STEP 6: MULTI-QUERY EMBEDDING


In [ ]:

# Embed all enhanced queries
print(f"🔢 Generating embeddings for {len(enhanced_queries)} queries...")
enhanced_query_embeddings = embedding_model.encode(
    enhanced_queries,
    convert_to_numpy=True,
    show_progress_bar=False
)

print(f"✅ Enhanced query embeddings shape: {enhanced_query_embeddings.shape}")


🔢 Generating embeddings for 6 queries...
✅ Enhanced query embeddings shape: (6, 384)


# STEP 7: RETRIEVAL (Cosine Similarity)


In [ ]:

all_similarities = []
for i, query_emb in enumerate(enhanced_query_embeddings):
    query_emb_reshaped = query_emb.reshape(1, -1)
    similarities = cosine_similarity(query_emb_reshaped, chunk_embeddings)[0]
    all_similarities.append(similarities)
    print(f"  ✅ Query {i+1}/{len(enhanced_queries)} similarities calculated")

# Aggregate similarities (take maximum across all queries for each chunk)
print("\n📊 Aggregating similarities across all queries...")
aggregated_similarities = np.max(all_similarities, axis=0)

# Convert to distances
distances = 1 - aggregated_similarities

# Get top K results
print(f"🎯 Retrieving top {TOP_K_RETRIEVE} chunks based on aggregated scores...")
top_indices = np.argsort(distances)[:TOP_K_RETRIEVE]

retrieved_chunks = [chunks[i] for i in top_indices]
retrieved_distances = [distances[i] for i in top_indices]

print(f"\n✅ Retrieved {len(retrieved_chunks)} chunks (multi-query retrieval):")
for i, (chunk, dist) in enumerate(zip(retrieved_chunks, retrieved_distances), 1):
    preview = chunk[:80] + "..." if len(chunk) > 80 else chunk
    print(f"  {i}. Distance: {dist:.4f} | {preview}")

  ✅ Query 1/6 similarities calculated
  ✅ Query 2/6 similarities calculated
  ✅ Query 3/6 similarities calculated
  ✅ Query 4/6 similarities calculated
  ✅ Query 5/6 similarities calculated
  ✅ Query 6/6 similarities calculated

📊 Aggregating similarities across all queries...
🎯 Retrieving top 10 chunks based on aggregated scores...

✅ Retrieved 10 chunks (multi-query retrieval):
  1. Distance: 0.2495 | extractive tasks, we ﬁnd that unconstrained generation outperforms previous extr...
  2. Distance: 0.2791 | pieces of retrieved content, as well as learning latent retrieval, and retrievin...
  3. Distance: 0.2845 | decisions and updating their world knowledge remain open research problems. Pre-...
  4. Distance: 0.3032 | 4.5 Additional Results
Generation Diversity Section 4.3 shows that RAG models ar...
  5. Distance: 0.3162 | RAG models can go beyond simple extractive QA and answer questions with free-for...
  6. Distance: 0.3216 | treatz as a latent variable and marginalize over seq2

# STEP 8: FILTERING BY DISTANCE THRESHOLD


In [ ]:
filtered_chunks = []
filtered_distances = []

for chunk, dist in zip(retrieved_chunks, retrieved_distances):
    if dist < DISTANCE_THRESHOLD:
        filtered_chunks.append(chunk)
        filtered_distances.append(dist)

print(f"Filtered down to {len(filtered_chunks)} relevant chunks")

if not filtered_chunks:
    print("No chunks passed the distance threshold")
    print("Continuing with all retrieved chunks...")
    filtered_chunks = retrieved_chunks
    filtered_distances = retrieved_distances

Filtered down to 10 relevant chunks


# STEP 9: RERANKING (CrossEncoder)

In [ ]:

# Load reranker model
print(f"Loading CrossEncoder: {RERANKER_MODEL}")
reranker = CrossEncoder(RERANKER_MODEL)

# Create query-document pairs
print(f"Reranking {len(filtered_chunks)} chunks...")
pairs = [[query, chunk] for chunk in filtered_chunks]

# Get reranking scores
rerank_scores = reranker.predict(pairs)

# Sort by reranking scores (higher is better)
reranked_indices = np.argsort(rerank_scores)[::-1][:TOP_K_RERANK]

reranked_chunks = [filtered_chunks[i] for i in reranked_indices]
reranked_scores = [rerank_scores[i] for i in reranked_indices]
reranked_distances = [filtered_distances[i] for i in reranked_indices]

print(f"\nTop {len(reranked_chunks)} reranked chunks:")
for i, (chunk, score, dist) in enumerate(zip(reranked_chunks, reranked_scores, reranked_distances), 1):
    preview = chunk[:80] + "..." if len(chunk) > 80 else chunk
    print(f"  {i}. Score: {score:.4f}, Dist: {dist:.4f} | {preview}")

Loading CrossEncoder: cross-encoder/ms-marco-MiniLM-L-6-v2


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Reranking 10 chunks...

Top 5 reranked chunks:
  1. Score: 3.0131, Dist: 0.3267 | 4 Results
4.1 Open-domain Question Answering
Table 1 shows results for RAG along...
  2. Score: 2.3259, Dist: 0.2845 | decisions and updating their world knowledge remain open research problems. Pre-...
  3. Score: 1.3891, Dist: 0.3162 | RAG models can go beyond simple extractive QA and answer questions with free-for...
  4. Score: 0.8146, Dist: 0.2791 | pieces of retrieved content, as well as learning latent retrieval, and retrievin...
  5. Score: -0.1739, Dist: 0.3032 | 4.5 Additional Results
Generation Diversity Section 4.3 shows that RAG models ar...


# STEP 10: CONTEXT PREPARATION


In [ ]:
# Format context from reranked chunks
context_parts = []
for i, (chunk, score) in enumerate(zip(reranked_chunks, reranked_scores), 1):
    context_parts.append(f"[Document {i}, Score: {score:.2f}]\n{chunk}\n")

context = "\n".join(context_parts)

print(f"Context prepared with {len(reranked_chunks)} chunks")
print(f"Total context length: {len(context):,} characters")

Context prepared with 5 chunks
Total context length: 4,891 characters


# STEP 11: LLM GENERATION (Google Gemini)


In [ ]:
# Configure Gemini
print(f"Configuring Gemini API...")
# Create prompt
prompt = f"""You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question.

If you don't know the answer based on the context, say that you don't know.
Use three sentences maximum and keep the answer concise.

Context:
{context}

Question: {query}

Answer:"""

print(f"Generating answer with {GEMINI_MODEL}...")
print(f"Prompt length: {len(prompt):,} characters")

# Generate response
response = client.models.generate_content(
            model=GEMINI_MODEL,
            contents=prompt
        )
answer = response.text

print(f"\n{'='*70}")
print("ANSWER:")
print('='*70)
print(answer)
print('='*70)

Configuring Gemini API...
Generating answer with gemini-2.5-flash...
Prompt length: 5,188 characters

ANSWER:
Retrieval-Augmented Generation (RAG) models combine pre-trained parametric and non-parametric memory for language generation. These models typically use a pre-trained seq2seq model as parametric memory and a dense vector index of Wikipedia, accessed by a neural retriever, as non-parametric memory. RAG can go beyond simple extractive QA, enabling free-form, abstractive text generation and achieving state-of-the-art results on open-domain QA tasks.


# STEP 12: UMAP VISUALIZATION


In [37]:
import numpy as np
import umap
import plotly.express as px
import pandas as pd

print("🗺️ Creating 3D UMAP (Dark + Retrieval Highlight)...")

# Combine embeddings
all_embeddings = np.vstack([chunk_embeddings, enhanced_query_embeddings])

# ---- LABELS ----
query_labels = ["Original Query", "Sub Query", "Sub Query", "Sub Query", "HyDE", "Step-back"]

labels = ["Document"] * len(chunk_embeddings)

for i in range(len(enhanced_query_embeddings)):
    if i < len(query_labels):
        labels.append(query_labels[i])
    else:
        labels.append("Query")

# ---- MARK RETRIEVED DOCS ----
# top_indices = indices of retrieved chunks (you already have this)
retrieved_set = set(top_indices)

for i in range(len(chunk_embeddings)):
    if i in retrieved_set:
        labels[i] = "Retrieved Doc"

# ---- UMAP ----
reducer = umap.UMAP(
    n_components=3,
    n_neighbors=min(10, len(all_embeddings)-1),
    random_state=42,
    metric="cosine"
)

embeddings_3d = reducer.fit_transform(all_embeddings)

# ---- DATAFRAME ----
df = pd.DataFrame({
    "x": embeddings_3d[:, 0],
    "y": embeddings_3d[:, 1],
    "z": embeddings_3d[:, 2],
    "type": labels
})

# ---- COLOR MAP ----
color_map = {
    "Document": "#4CC9F0",        # light blue
    "Retrieved Doc": "#1D3557",   # dark blue (highlight)
    "Original Query": "#F72585",
    "Sub Query": "#FCA311",
    "HyDE": "#7209B7",
    "Step-back": "#2EC4B6",
    "Query": "#AAAAAA"
}

# ---- SIZE MAP ----
size_map = {
    "Document": 4,
    "Retrieved Doc": 6,
    "Original Query": 10,
    "Sub Query": 7,
    "HyDE": 9,
    "Step-back": 9,
    "Query": 6
}

df["size"] = df["type"].map(size_map)

# ---- PLOT ----
symbol_map = {
    "Original Query": "x",
    "Sub Query": "square",
    "HyDE": "square",
    "Step-back": "square",
    "Retrieved Doc": "circle-open",
    "Document": "circle"
}

fig = px.scatter_3d(
    df,
    x="x",
    y="y",
    z="z",
    color="type",
    size="size",
    symbol="type",
    symbol_map=symbol_map,
    color_discrete_map=color_map,
    template="plotly_dark",
    title="3D UMAP"

)

# Layout
fig.update_layout(
    scene=dict(
        xaxis_title="UMAP 1",
        yaxis_title="UMAP 2",
        zaxis_title="UMAP 3"
    ),
    legend=dict(bgcolor="rgba(0,0,0,0.5)")
)

fig.show()

🗺️ Creating 3D UMAP (Dark + Retrieval Highlight)...


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



In [38]:
output_file = "rag_pipeline_umap.html"
fig.write_html(output_file)
print(f"\n  💾 Saved to: {output_file}")


  💾 Saved to: rag_pipeline_umap.html


# STEP 12: SUMMARY


In [ ]:
print("\n" + "="*70)
print("📊 PIPELINE SUMMARY")
print("="*70)

print(f"""
📄 Document: {Path(PDF_FILE).name}
  └─ Pages: {total_pages}
  └─ Characters: {len(full_text):,}

✂️ Chunking:
  └─ Total chunks: {len(chunks)}
  └─ Chunk size: {CHUNK_SIZE}
  └─ Overlap: {CHUNK_OVERLAP}

🔢 Embeddings:
  └─ Model: {EMBEDDING_MODEL}
  └─ Dimension: {chunk_embeddings.shape[1]}

🔍 Query: "{query}"

📊 Retrieval:
  └─ Retrieved: {len(retrieved_chunks)} chunks
  └─ Filtered: {len(filtered_chunks)} chunks (distance < {DISTANCE_THRESHOLD})

🔄 Reranking:
  └─ Model: {RERANKER_MODEL}
  └─ Top reranked: {len(reranked_chunks)} chunks

🤖 Generation:
  └─ Model: {GEMINI_MODEL}
  └─ Context length: {len(context):,} characters
  └─ Answer length: {len(answer):,} characters

✅ Pipeline completed successfully!
""")

print("="*70)
print("🎉 ALL STEPS COMPLETED!")
print("="*70 + "\n")

In [ ]:
"""
Complete RAG Pipeline with UMAP Visualization
All steps in linear order - No functions, just sequential code
Uses: PyPDF, LangChain, SentenceTransformer, CrossEncoder, Gemini, UMAP
"""

import os
from pathlib import Path
from dotenv import load_dotenv
import numpy as np
import plotly.graph_objects as go
from pypdf import PdfReader
from google import genai
from sentence_transformers import CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
# Load environment variables
load_dotenv()

from google.colab import userdata
# ============================================================================
# STEP 1: CONFIGURATION
# ============================================================================

print("\n" + "="*70)
print("📚 COMPLETE RAG PIPELINE WITH UMAP VISUALIZATION")
print("="*70 + "\n")

# File to process
PDF_FILE = "/content/rag2020.pdf"  # ← CHANGE THIS TO YOUR PDF

# API Keys
GEMINI_API_KEY = userdata.get('Gemini_key') # or GEMINI_LLM_KEY

# Settings
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
TOP_K_RETRIEVE = 10
TOP_K_RERANK = 5
DISTANCE_THRESHOLD = 1.5

# Models
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
RERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
GEMINI_MODEL = "gemini-2.5-flash"

print(f" Document: {PDF_FILE}")
print(f" Embedding Model: {EMBEDDING_MODEL}")
print(f" Reranker Model: {RERANKER_MODEL}")
print(f" LLM Model: {GEMINI_MODEL}")
print(f" Chunk Size: {CHUNK_SIZE} chars, Overlap: {CHUNK_OVERLAP}")

# ============================================================================
# STEP 2: DOCUMENT LOADING (PyPDF)
# ============================================================================

print("\n" + "="*70)
print("STEP 1: LOADING DOCUMENT")
print("="*70)


# Check if file exists
if not Path(PDF_FILE).exists():
    print(f"❌ File not found: {PDF_FILE}")
    print("Please update the PDF_FILE variable at the top of the script")
    exit()

# Load PDF
print(f"📖 Reading PDF: {PDF_FILE}")
pdf_reader = PdfReader(PDF_FILE)
total_pages = len(pdf_reader.pages)

# Extract text from all pages
full_text = ""
for page_num, page in enumerate(pdf_reader.pages, 1):
    page_text = page.extract_text()
    if page_text:
        full_text += page_text + "\n"
    print(f"  ✅ Processed page {page_num}/{total_pages}")

print(f"\n✅ Loaded {len(full_text):,} characters from {total_pages} pages")

# ============================================================================
# STEP 3: CHUNKING (LangChain Text Splitter)
# ============================================================================

print("\n" + "="*70)
print("STEP 2: CHUNKING TEXT")
print("="*70)

# Create text splitter
print(f"Creating chunks (size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP})")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ".", "!", "?", " ", ""],
    length_function=len
)

# Split text into chunks
chunks = text_splitter.split_text(full_text)

print(f"Created {len(chunks)} chunks")
print(f"\n Sample chunks:")
for i, chunk in enumerate(chunks[:3], 1):
    preview = chunk[:100] + "..." if len(chunk) > 100 else chunk
    print(f"  Chunk {i}: {preview}")

# ============================================================================
# STEP 4: EMBEDDING (SentenceTransformer)
# ============================================================================

print("\n" + "="*70)
print("STEP 3: GENERATING EMBEDDINGS")
print("="*70)


# Load embedding model
print(f"Loading SentenceTransformer: {EMBEDDING_MODEL}")
embedding_model = SentenceTransformer(EMBEDDING_MODEL)

# Generate embeddings for all chunks
print(f"Generating embeddings for {len(chunks)} chunks...")
chunk_embeddings = embedding_model.encode(
    chunks,
    convert_to_numpy=True,
    show_progress_bar=True
)

print(f"Generated embeddings with shape: {chunk_embeddings.shape}")
print(f"Embedding dimension: {chunk_embeddings.shape[1]}")

# ============================================================================
# STEP 5: QUERY PROCESSING
# ============================================================================

print("\n" + "="*70)
print("STEP 4: QUERY PROCESSING")
print("="*70)

# Get query from user
query = input("\nEnter your question: ").strip()

if not query:
    query = "What is this document about?"
    print(f"Using default query: {query}")

print(f"\n🔍 Query: {query}")

# Embed the query
print("Generating query embedding...")
query_embedding = embedding_model.encode([query], convert_to_numpy=True)

print(f"Query embedding shape: {query_embedding.shape}")

# ============================================================================
# STEP 6: RETRIEVAL (Cosine Similarity)
# ============================================================================

print("\n" + "="*70)
print("STEP 5: RETRIEVING RELEVANT CHUNKS")
print("="*70)


# Calculate cosine similarity between query and all chunks
print(f"Calculating similarity with {len(chunks)} chunks...")
similarities = cosine_similarity(query_embedding, chunk_embeddings)[0]

# Convert to distances (1 - similarity for consistency with ChromaDB)
distances = 1 - similarities

# Get top K results
print(f"Retrieving top {TOP_K_RETRIEVE} chunks...")
top_indices = np.argsort(distances)[:TOP_K_RETRIEVE]

retrieved_chunks = [chunks[i] for i in top_indices]
retrieved_distances = [distances[i] for i in top_indices]

print(f"\nRetrieved {len(retrieved_chunks)} chunks:")
for i, (chunk, dist) in enumerate(zip(retrieved_chunks, retrieved_distances), 1):
    preview = chunk[:80] + "..." if len(chunk) > 80 else chunk
    print(f"  {i}. Distance: {dist:.4f} | {preview}")

# ============================================================================
# STEP 7: FILTERING BY DISTANCE THRESHOLD
# ============================================================================

print("\n" + "="*70)
print("STEP 6: FILTERING BY DISTANCE THRESHOLD")
print("="*70)

print(f"Filtering chunks with distance < {DISTANCE_THRESHOLD}")

filtered_chunks = []
filtered_distances = []

for chunk, dist in zip(retrieved_chunks, retrieved_distances):
    if dist < DISTANCE_THRESHOLD:
        filtered_chunks.append(chunk)
        filtered_distances.append(dist)

print(f"Filtered down to {len(filtered_chunks)} relevant chunks")

if not filtered_chunks:
    print("No chunks passed the distance threshold")
    print("Continuing with all retrieved chunks...")
    filtered_chunks = retrieved_chunks
    filtered_distances = retrieved_distances

# ============================================================================
# STEP 8: RERANKING (CrossEncoder)
# ============================================================================

print("\n" + "="*70)
print("STEP 7: RERANKING WITH CROSS-ENCODER")
print("="*70)


# Load reranker model
print(f"Loading CrossEncoder: {RERANKER_MODEL}")
reranker = CrossEncoder(RERANKER_MODEL)

# Create query-document pairs
print(f"Reranking {len(filtered_chunks)} chunks...")
pairs = [[query, chunk] for chunk in filtered_chunks]

# Get reranking scores
rerank_scores = reranker.predict(pairs)

# Sort by reranking scores (higher is better)
reranked_indices = np.argsort(rerank_scores)[::-1][:TOP_K_RERANK]

reranked_chunks = [filtered_chunks[i] for i in reranked_indices]
reranked_scores = [rerank_scores[i] for i in reranked_indices]
reranked_distances = [filtered_distances[i] for i in reranked_indices]

print(f"\nTop {len(reranked_chunks)} reranked chunks:")
for i, (chunk, score, dist) in enumerate(zip(reranked_chunks, reranked_scores, reranked_distances), 1):
    preview = chunk[:80] + "..." if len(chunk) > 80 else chunk
    print(f"  {i}. Score: {score:.4f}, Dist: {dist:.4f} | {preview}")

# ============================================================================
# STEP 9: CONTEXT PREPARATION
# ============================================================================

print("\n" + "="*70)
print("STEP 8: PREPARING CONTEXT FOR LLM")
print("="*70)

# Format context from reranked chunks
context_parts = []
for i, (chunk, score) in enumerate(zip(reranked_chunks, reranked_scores), 1):
    context_parts.append(f"[Document {i}, Score: {score:.2f}]\n{chunk}\n")

context = "\n".join(context_parts)

print(f"Context prepared with {len(reranked_chunks)} chunks")
print(f"Total context length: {len(context):,} characters")

# ============================================================================
# STEP 10: LLM GENERATION (Google Gemini)
# ============================================================================

print("\n" + "="*70)
print("STEP 9: GENERATING ANSWER WITH GEMINI")
print("="*70)


# Configure Gemini
print(f"Configuring Gemini API...")
client = genai.Client(api_key=GEMINI_API_KEY)
# Create prompt
prompt = f"""You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question.

If you don't know the answer based on the context, say that you don't know.
Use three sentences maximum and keep the answer concise.

Context:
{context}

Question: {query}

Answer:"""

print(f"Generating answer with {GEMINI_MODEL}...")
print(f"Prompt length: {len(prompt):,} characters")

# Generate response
response = client.models.generate_content(
            model=GEMINI_MODEL,
            contents=prompt
        )
answer = response.text

print(f"\n{'='*70}")
print("ANSWER:")
print('='*70)
print(answer)
print('='*70)

# ============================================================================
# STEP 11: UMAP VISUALIZATION
# ============================================================================

print("\n" + "="*70)
print("STEP 10: CREATING UMAP VISUALIZATION")
print("="*70)

import umap

# Ask if user wants UMAP
show_umap = input("\n🗺️ Show UMAP visualization? (y/n): ").strip().lower()

if show_umap == 'y':

    print("\n Creating UMAP visualization...")

    # Prepare all embeddings (chunks + query)
    all_embeddings = np.vstack([chunk_embeddings, query_embedding])

    # Create labels
    labels = ["Document"] * len(chunks)
    labels.append("Query")

    # Create texts for hover
    texts = chunks.copy()
    texts.append(query)

    # Create colors
    colors = ["lightblue"] * len(chunks)
    colors.append("red")

    # Create sizes
    sizes = [6] * len(chunks)
    sizes.append(18)

    # Create symbols
    symbols = ["circle"] * len(chunks)
    symbols.append("star")

    # Mark retrieved chunks
    print(" Marking retrieved chunks...")
    retrieved_chunk_indices = top_indices[:len(filtered_chunks)]
    for idx in retrieved_chunk_indices:
        colors[idx] = "darkblue"
        sizes[idx] = 10
        symbols[idx] = "diamond"
        labels[idx] = "Retrieved"

    # Apply UMAP
    print("Running UMAP dimensionality reduction...")
    n_neighbors = min(15, len(all_embeddings) - 1)

    reducer = umap.UMAP(
        n_neighbors=n_neighbors,
        min_dist=0.1,
        random_state=42,
        metric="cosine",
        verbose=True
    )

    embeddings_2d = reducer.fit_transform(all_embeddings)

    print("UMAP reduction complete")

    # Create Plotly figure
    print(" Creating visualization...")
    fig = go.Figure()

    # Group by type for legend
    for label_type in set(labels):
        mask = [label == label_type for label in labels]
        indices = [i for i, m in enumerate(mask) if m]

        if not indices:
            continue

        x_coords = [embeddings_2d[i, 0] for i in indices]
        y_coords = [embeddings_2d[i, 1] for i in indices]
        hover_texts = [f"{labels[i]}: {texts[i][:120]}..." for i in indices]
        point_colors = [colors[i] for i in indices]
        point_sizes = [sizes[i] for i in indices]
        point_symbols = [symbols[i] for i in indices]

        mode = 'markers+text' if label_type == "Query" else 'markers'
        text = [label_type] if label_type == "Query" else None
        textposition = 'top center' if label_type == "Query" else None

        fig.add_trace(go.Scatter(
            x=x_coords,
            y=y_coords,
            mode=mode,
            marker=dict(
                size=point_sizes,
                color=point_colors,
                symbol=point_symbols,
                line=dict(width=1, color='black'),
                opacity=0.8
            ),
            text=text,
            textposition=textposition,
            hovertext=hover_texts,
            hoverinfo='text',
            name=f"{label_type} ({len(indices)})"
        ))

    # Update layout
    query_preview = query[:60] + "..." if len(query) > 60 else query

    fig.update_layout(
        title=dict(
            text=f"📊 RAG Pipeline UMAP Visualization<br><sub>Query: '{query_preview}'</sub>",
            x=0.5,
            xanchor='center'
        ),
        xaxis_title="UMAP Dimension 1",
        yaxis_title="UMAP Dimension 2",
        hovermode="closest",
        height=800,
        width=1200,
        showlegend=True,
        legend=dict(
            x=0.01,
            y=0.99,
            bgcolor="rgba(255,255,255,0.9)",
            bordercolor="black",
            borderwidth=1
        ),
        plot_bgcolor='white',
        xaxis=dict(gridcolor='lightgray', showgrid=True),
        yaxis=dict(gridcolor='lightgray', showgrid=True)
    )

    # Save and show
    fig.show()
    output_file = "rag_pipeline_umap.html"
    fig.write_html(output_file)
    print(f"\n  💾 Saved to: {output_file}")

    print("  🌐 Opening in browser...")

    print("\n✅ UMAP visualization complete!")
    print("\n📖 Legend:")
    print("  🔵 Light blue circles = All document chunks")
    print("  🔴 Red star = Your query")
    print("  🔷 Dark blue diamonds = Retrieved chunks")

# ============================================================================
# STEP 12: SUMMARY
# ============================================================================

print("\n" + "="*70)
print("📊 PIPELINE SUMMARY")
print("="*70)

print(f"""
📄 Document: {Path(PDF_FILE).name}
  └─ Pages: {total_pages}
  └─ Characters: {len(full_text):,}

✂️ Chunking:
  └─ Total chunks: {len(chunks)}
  └─ Chunk size: {CHUNK_SIZE}
  └─ Overlap: {CHUNK_OVERLAP}

🔢 Embeddings:
  └─ Model: {EMBEDDING_MODEL}
  └─ Dimension: {chunk_embeddings.shape[1]}

🔍 Query: "{query}"

📊 Retrieval:
  └─ Retrieved: {len(retrieved_chunks)} chunks
  └─ Filtered: {len(filtered_chunks)} chunks (distance < {DISTANCE_THRESHOLD})

🔄 Reranking:
  └─ Model: {RERANKER_MODEL}
  └─ Top reranked: {len(reranked_chunks)} chunks

🤖 Generation:
  └─ Model: {GEMINI_MODEL}
  └─ Context length: {len(context):,} characters
  └─ Answer length: {len(answer):,} characters

✅ Pipeline completed successfully!
""")

print("="*70)
print("🎉 ALL STEPS COMPLETED!")
print("="*70 + "\n")


📚 COMPLETE RAG PIPELINE WITH UMAP VISUALIZATION

 Document: /content/rag2020.pdf
 Embedding Model: all-MiniLM-L6-v2
 Reranker Model: cross-encoder/ms-marco-MiniLM-L-6-v2
 LLM Model: gemini-2.5-flash
 Chunk Size: 1000 chars, Overlap: 200

STEP 1: LOADING DOCUMENT
📖 Reading PDF: /content/rag2020.pdf
  ✅ Processed page 1/19
  ✅ Processed page 2/19
  ✅ Processed page 3/19
  ✅ Processed page 4/19
  ✅ Processed page 5/19
  ✅ Processed page 6/19
  ✅ Processed page 7/19
  ✅ Processed page 8/19
  ✅ Processed page 9/19
  ✅ Processed page 10/19
  ✅ Processed page 11/19
  ✅ Processed page 12/19
  ✅ Processed page 13/19
  ✅ Processed page 14/19
  ✅ Processed page 15/19
  ✅ Processed page 16/19
  ✅ Processed page 17/19
  ✅ Processed page 18/19
  ✅ Processed page 19/19

✅ Loaded 69,075 characters from 19 pages

STEP 2: CHUNKING TEXT
Creating chunks (size=1000, overlap=200)
Created 88 chunks

 Sample chunks:
  Chunk 1: Retrieval-Augmented Generation for
Knowledge-Intensive NLP Tasks
Patrick Lewis†‡, 

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generating embeddings for 88 chunks...


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Generated embeddings with shape: (88, 384)
Embedding dimension: 384

STEP 4: QUERY PROCESSING

Enter your question: what is rag

🔍 Query: what is rag
Generating query embedding...
Query embedding shape: (1, 384)

STEP 5: RETRIEVING RELEVANT CHUNKS
Calculating similarity with 88 chunks...
Retrieving top 10 chunks...

Retrieved 10 chunks:
  1. Distance: 0.5590 | This shows we can update RAG’s world knowledge by simply replacing its non-param...
  2. Distance: 0.6186 | with both models outperforming BART on Q-BLEU-1. 4 shows human evaluation result...
  3. Distance: 0.6444 | RAG-T Dante’s "Inferno" is the ﬁrst part of this epic poem
RAG-S This 14th centu...
  4. Distance: 0.6468 | RAG models can go beyond simple extractive QA and answer questions with free-for...
  5. Distance: 0.6558 | Exact Match B-1 QB-1 R-L B-1 Label Accuracy
RAG-Token-BM25 29.7 41.5 32.1 33.1 1...
  6. Distance: 0.6589 | as a target sequence of length one, in which case RAG-Sequence and RAG-Token are...
  7. Distance

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Reranking 10 chunks...

Top 5 reranked chunks:
  1. Score: 2.4641, Dist: 0.6444 | RAG-T Dante’s "Inferno" is the ﬁrst part of this epic poem
RAG-S This 14th centu...
  2. Score: 1.3891, Dist: 0.6468 | RAG models can go beyond simple extractive QA and answer questions with free-for...
  3. Score: 0.4778, Dist: 0.6692 | points and 2.6 Rouge-L points. RAG approaches state-of-the-art model performance...
  4. Score: 0.1143, Dist: 0.6186 | with both models outperforming BART on Q-BLEU-1. 4 shows human evaluation result...
  5. Score: -1.2593, Dist: 0.5590 | This shows we can update RAG’s world knowledge by simply replacing its non-param...

STEP 8: PREPARING CONTEXT FOR LLM
Context prepared with 5 chunks
Total context length: 4,891 characters

STEP 9: GENERATING ANSWER WITH GEMINI
Configuring Gemini API...
Generating answer with gemini-2.5-flash...
Prompt length: 5,190 characters

ANSWER:
RAG models are designed for abstractive text generation and can answer questions in a knowledge-intensi